In [7]:
#imports
import os
import yt_dlp
import librosa
import numpy as np
import pandas as pd
from typing import Optional

Genres and subgenres in part 2

Classes: 12 — ['Classical/Instrumental', 'Country/Folk', 'Electronic', 'Hip-Hop/R&B', 'House/Dance', 'Jazz/Blues', 'Latin', 'Metal', 'Pop', 'Reggae', 'Rock', 'World/Other']

In [8]:
# map every subgenre to a parent genre

GENRE_MAP = {
    # Rock
    'rock': 'Rock', 'alt-rock': 'Rock', 'punk-rock': 'Rock',
    'hard-rock': 'Rock', 'psych-rock': 'Rock', 'grunge': 'Rock',
    'rock-n-roll': 'Rock', 'emo': 'Rock', 'indie': 'Rock',
    'alternative': 'Rock', 'rockabilly': 'Rock',

    # Metal
    'metal': 'Metal', 'heavy-metal': 'Metal', 'death-metal': 'Metal',
    'black-metal': 'Metal', 'metalcore': 'Metal', 'grindcore': 'Metal',
    'hardcore': 'Metal', 'goth': 'Metal', 'industrial': 'Metal',

    # Electronic
    'edm': 'Electronic', 'electronic': 'Electronic', 'techno': 'Electronic',
    'detroit-techno': 'Electronic', 'minimal-techno': 'Electronic',
    'trance': 'Electronic', 'dubstep': 'Electronic', 'drum-and-bass': 'Electronic',
    'idm': 'Electronic', 'electro': 'Electronic', 'breakbeat': 'Electronic',
    'hardstyle': 'Electronic', 'ambient': 'Electronic',

    # House/Dance
    'house': 'House/Dance', 'deep-house': 'House/Dance',
    'chicago-house': 'House/Dance', 'progressive-house': 'House/Dance',
    'dance': 'House/Dance', 'disco': 'House/Dance', 'club': 'House/Dance',
    'garage': 'House/Dance', 'dancehall': 'House/Dance',

    # Pop
    'pop': 'Pop', 'indie-pop': 'Pop', 'synth-pop': 'Pop',
    'power-pop': 'Pop', 'k-pop': 'Pop', 'j-pop': 'Pop',
    'cantopop': 'Pop', 'mandopop': 'Pop', 'j-idol': 'Pop',
    'j-dance': 'Pop', 'party': 'Pop', 'happy': 'Pop',
    'pop-film': 'Pop', 'disney': 'Pop', 'show-tunes': 'Pop',

    # Hip-Hop / R&B
    'hip-hop': 'Hip-Hop/R&B', 'r-n-b': 'Hip-Hop/R&B', 'soul': 'Hip-Hop/R&B',
    'funk': 'Hip-Hop/R&B', 'groove': 'Hip-Hop/R&B', 'trip-hop': 'Hip-Hop/R&B',

    # Latin
    'latin': 'Latin', 'latino': 'Latin', 'salsa': 'Latin',
    'samba': 'Latin', 'reggaeton': 'Latin', 'tango': 'Latin',
    'forro': 'Latin', 'pagode': 'Latin', 'sertanejo': 'Latin',
    'mpb': 'Latin', 'brazil': 'Latin', 'bossanova': 'Latin',
    'romance': 'Latin', 'spanish': 'Latin',

    # Jazz / Blues
    'jazz': 'Jazz/Blues', 'blues': 'Jazz/Blues', 'gospel': 'Jazz/Blues',
    'soul': 'Jazz/Blues',

    # Classical / Instrumental
    'classical': 'Classical/Instrumental', 'opera': 'Classical/Instrumental',
    'piano': 'Classical/Instrumental', 'guitar': 'Classical/Instrumental',
    'new-age': 'Classical/Instrumental', 'sleep': 'Classical/Instrumental',
    'study': 'Classical/Instrumental',

    # Country / Folk
    'country': 'Country/Folk', 'folk': 'Country/Folk', 'bluegrass': 'Country/Folk',
    'honky-tonk': 'Country/Folk', 'singer-songwriter': 'Country/Folk',
    'songwriter': 'Country/Folk', 'acoustic': 'Country/Folk',

    # Reggae
    'reggae': 'Reggae', 'dub': 'Reggae', 'ska': 'Reggae',

    # World / Other
    'world-music': 'World/Other', 'afrobeat': 'World/Other',
    'indian': 'World/Other', 'iranian': 'World/Other',
    'turkish': 'World/Other', 'malay': 'World/Other',
    'french': 'World/Other', 'german': 'World/Other',
    'swedish': 'World/Other', 'british': 'World/Other',
    'j-rock': 'World/Other', 'anime': 'World/Other',
    'children': 'World/Other', 'kids': 'World/Other',
    'comedy': 'World/Other', 'sad': 'World/Other',
    'chill': 'World/Other', 'dub': 'World/Other',
}

In [9]:
#load data
data = pd.read_csv('/Users/Owner/Desktop/DSCourses/DS 340/340_Project/DS-340-Group-Project/Data/spotify-tracks-dataset.csv')

In [10]:
#helper functions



MIN_SAMPLES_PER_GENRE = 50
SAMPLES_PER_SUBGENRE  = 10


def build_genre_to_subgenres(genre_map: dict) -> dict[str, list[str]]:
    """Invert GENRE_MAP: genre label -> list of subgenre keys."""
    genre_to_subs = {}
    for subgenre, genre in genre_map.items():
        genre_to_subs.setdefault(genre, []).append(subgenre)
    return genre_to_subs


def compute_sampling_plan(
    genre_to_subs: dict[str, list[str]],
    min_per_genre: int = MIN_SAMPLES_PER_GENRE,
    base_per_subgenre: int = SAMPLES_PER_SUBGENRE,
) -> dict[str, dict[str, int]]:
    """
    Returns a plan: { genre: { subgenre: n_samples } }

    Logic:
      - Natural allocation = base_per_subgenre * n_subgenres
      - If natural allocation < min_per_genre, scale up evenly so the
        total hits the floor (distributed as evenly as possible).
    """
    plan = {}
    for genre, subgenres in genre_to_subs.items():
        n_subs = len(subgenres)
        natural_total = base_per_subgenre * n_subs

        if natural_total >= min_per_genre:
            per_sub = base_per_subgenre
            remainder = 0
        else:
            per_sub, remainder = divmod(min_per_genre, n_subs)

        allocation = {}
        for i, sub in enumerate(subgenres):
            # Distribute the remainder 1-extra to the first `remainder` subgenres
            allocation[sub] = per_sub + (1 if i < remainder else 0)

        plan[genre] = allocation
    return plan


def sample_tracks(
    plan: dict[str, dict[str, int]],
    available_tracks: dict[str, list],
) -> dict[str, dict[str, list]]:
    """
    Given a sampling plan and a dict of { subgenre: [track, ...] },
    randomly sample the requested number of tracks per subgenre.

    Returns { genre: { subgenre: [sampled tracks] } }
    """
    sampled = {}
    for genre, subgenre_counts in plan.items():
        sampled[genre] = {}
        for subgenre, n in subgenre_counts.items():
            tracks = available_tracks.get(subgenre, [])
            k = min(n, len(tracks))  # don't crash if API returned fewer tracks
            if k < n:
                print(f"  [WARN] {subgenre} ({genre}): requested {n}, only {k} available")
            sampled[genre][subgenre] = random.sample(tracks, k) if k > 0 else []
    return sampled


def print_plan_summary(plan: dict[str, dict[str, int]]) -> None:
    print(f"\n{'Genre':<25} {'Subgenres':>9} {'Total':>7}")
    print("-" * 44)
    grand_total = 0
    for genre, alloc in sorted(plan.items()):
        total = sum(alloc.values())
        grand_total += total
        print(f"{genre:<25} {len(alloc):>9} {total:>7}")
    print("-" * 44)
    print(f"{'GRAND TOTAL':<25} {'':>9} {grand_total:>7}\n")

In [11]:
# mfcc data set sampling

import random

df = data

# Drop junk index columns
df = df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'])

# Map subgenres to parent genres
df['genre'] = df['track_genre'].map(GENRE_MAP)

# Drop tracks whose subgenre isn't in GENRE_MAP
df = df.dropna(subset=['genre'])

# Build sampling plan (from previous code)
genre_to_subs = build_genre_to_subgenres(GENRE_MAP)
plan = compute_sampling_plan(genre_to_subs)

# Sample tracks according to plan
sampled_frames = []

for genre, subgenre_counts in plan.items():
    for subgenre, n in subgenre_counts.items():
        subset = df[df['track_genre'] == subgenre]
        k = min(n, len(subset))
        if k < n:
            print(f"[WARN] {subgenre} ({genre}): requested {n}, only {k} available")
        sampled_frames.append(subset.sample(k, random_state=42))

sampled_df = pd.concat(sampled_frames).drop_duplicates(subset='track_id').reset_index(drop=True)

print(f"Total sampled tracks: {len(sampled_df)}")
print(sampled_df.groupby('genre')['track_id'].count())

sampled_df.to_csv('sampled_tracks.csv', index=False)


[WARN] bossanova (Latin): requested 10, only 0 available
Total sampled tracks: 1164
genre
Classical/Instrumental     70
Country/Folk               66
Electronic                130
Hip-Hop/R&B                50
House/Dance                90
Jazz/Blues                 50
Latin                     130
Metal                      90
Pop                       150
Reggae                     48
Rock                      110
World/Other               180
Name: track_id, dtype: int64


In [12]:
sampled_df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,...,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre,genre
0,2Wzl4bZLjqFUOeZaGrTaVJ,YUNGBLUD,Drippy Drippy,parents,0,172026,True,0.594,0.833,9,...,0,0.0507,0.00967,0.000000,0.300,0.599,82.040,4,rock,Rock
1,1UEj7RpdqH02grDvdxTKLP,All Time Low,Alternative Christmas 2022,"Merry Christmas, Kiss My Ass",0,199253,False,0.574,0.966,4,...,1,0.0685,0.01410,0.000000,0.430,0.788,128.034,4,rock,Rock
2,3SCYKwSFwlSe5xEvVlcP1W,George Strait,Country Christmas Greatest Hits,Jingle Bell Rock,1,132493,False,0.775,0.562,4,...,0,0.0299,0.46800,0.000000,0.174,0.770,121.034,4,rock,Rock
3,1NhPKVLsHhFUHIOZ32QnS2,OneRepublic,Waking Up,Secrets,76,224693,False,0.516,0.764,2,...,1,0.0366,0.07170,0.000000,0.115,0.376,148.021,4,rock,Rock
4,1ihSSnA4d0dcxQouy7gtNJ,Juanes,Retro Baladas,Es Por Ti,0,251120,False,0.694,0.761,4,...,1,0.0263,0.17600,0.000036,0.123,0.866,129.953,4,rock,Rock


In [13]:
#working with spotify api to get previews --> spotify api doesnt let you get the song preview anymore, (ignore this codebloack)
%pip install spotipy

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import pandas as pd
import time

sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id='197675771a4247b1ab8f3e03b0a34e50',
    client_secret='6eca677adf6e41f2a34f163da9e76f40'
))

def get_preview_urls(track_ids: list[str], batch_size: int = 50) -> dict:
    """Fetch preview URLs in batches of 50 (Spotify API limit)."""
    preview_map = {}
    for i in range(0, len(track_ids), batch_size):
        batch = track_ids[i:i + batch_size]
        try:
            results = sp.tracks(batch)
            for track in results['tracks']:
                if track:
                    preview_map[track['id']] = track['preview_url']  # None if unavailable
        except Exception as e:
            print(f"[ERROR] Batch {i} failed: {e}")
        time.sleep(0.1)  # rate limit buffer
    return preview_map

sampled_df = pd.read_csv('sampled_tracks.csv')
preview_map = get_preview_urls(sampled_df['track_id'].tolist())

sampled_df['preview_url'] = sampled_df['track_id'].map(preview_map)

# How many previews are actually available
n_available = sampled_df['preview_url'].notna().sum()
n_total = len(sampled_df)
print(f"Preview URLs found: {n_available}/{n_total} ({n_available/n_total*100:.1f}%)")

sampled_df.to_csv('sampled_tracks_with_previews.csv', index=False)

  Using cached spotipy-2.26.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached redis-7.4.0-py3-none-any.whl.metadata (12 kB)
Using cached spotipy-2.26.0-py3-none-any.whl (31 kB)
Using cached redis-7.4.0-py3-none-any.whl (409 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [spotipy]
Note: you may need to restart the kernel to use updated packages.


HTTP Error for GET to https://api.spotify.com/v1/tracks/?ids=2Wzl4bZLjqFUOeZaGrTaVJ,1UEj7RpdqH02grDvdxTKLP,3SCYKwSFwlSe5xEvVlcP1W,1NhPKVLsHhFUHIOZ32QnS2,1ihSSnA4d0dcxQouy7gtNJ,333WK4wUSAfawJhirnJFAG,4dWWUYR9l8xKi9CqXCG8ke,57bgtoPSgt236HzfBOd8kj,7oAf0lBjipjSfyay9M8wQ6,5HNau9WX6OmX2PcBn3n24r,2GJ1xx5ZoeIJYbQKLSpoZ3,1MmzIqMP1y5AGIX20RZxDc,1rBivSsnwBbuC1UH9w9eCV,5d9RSSwDvlTyuwfmeafUah,0IK8y0vAO0sM630HsxwLht,44agmNQypTCFKc2dpVonDy,5izpR8PlJCjgJexf7JkMa9,0nkzd3yNniB767zSDDdLZ3,5YTUCHSwfwJ4oHFDjFgpEe,1955ZZJe1TzmSR0TomnNjI,2grJEu2P4KD1YDq2ZaaLoL,0SjveEILP1oTznahQcOJj5,5oayPLOM38NrQr8FzKa3os,1uT2b9tByCtdjL4PN3xbRs,4Pz6d9chtIKUQ628KWDZpR,4Haz8DCrtorRcaHKR1R8yo,4TmmjNjDVhTZ3pkanD3y7W,66jQuzmutqVNb7OUsXTdn9,3HM4NePuYAqdXpAGBk7nNz,0vGAY9aloPKORe0R0KlfGv,2vNw57KPaYDzkyPxXYUORX,5ij7PgYsUYHMXMFDWcZqww,2UsDkGHdQxiivgHhyTDtpd,2vyae4DvSU2gs8OMOGwXF2,2tAeN2TKlQLOoSPXtARzBV,0QSPohbvsYOEcvkHnok13V,5YnNWHKCJaJKwbagXDb5Uf,7n3WO6ESKS1uCI9fgkGs66,6kAOsnRUgp21bPiUoVZeuJ,1uOe9m9bAiAY4kGnGDu1Ns,5PffgchZakXQs8Err8E

[ERROR] Batch 0 failed: http status: 403, code: -1 - https://api.spotify.com/v1/tracks/?ids=2Wzl4bZLjqFUOeZaGrTaVJ,1UEj7RpdqH02grDvdxTKLP,3SCYKwSFwlSe5xEvVlcP1W,1NhPKVLsHhFUHIOZ32QnS2,1ihSSnA4d0dcxQouy7gtNJ,333WK4wUSAfawJhirnJFAG,4dWWUYR9l8xKi9CqXCG8ke,57bgtoPSgt236HzfBOd8kj,7oAf0lBjipjSfyay9M8wQ6,5HNau9WX6OmX2PcBn3n24r,2GJ1xx5ZoeIJYbQKLSpoZ3,1MmzIqMP1y5AGIX20RZxDc,1rBivSsnwBbuC1UH9w9eCV,5d9RSSwDvlTyuwfmeafUah,0IK8y0vAO0sM630HsxwLht,44agmNQypTCFKc2dpVonDy,5izpR8PlJCjgJexf7JkMa9,0nkzd3yNniB767zSDDdLZ3,5YTUCHSwfwJ4oHFDjFgpEe,1955ZZJe1TzmSR0TomnNjI,2grJEu2P4KD1YDq2ZaaLoL,0SjveEILP1oTznahQcOJj5,5oayPLOM38NrQr8FzKa3os,1uT2b9tByCtdjL4PN3xbRs,4Pz6d9chtIKUQ628KWDZpR,4Haz8DCrtorRcaHKR1R8yo,4TmmjNjDVhTZ3pkanD3y7W,66jQuzmutqVNb7OUsXTdn9,3HM4NePuYAqdXpAGBk7nNz,0vGAY9aloPKORe0R0KlfGv,2vNw57KPaYDzkyPxXYUORX,5ij7PgYsUYHMXMFDWcZqww,2UsDkGHdQxiivgHhyTDtpd,2vyae4DvSU2gs8OMOGwXF2,2tAeN2TKlQLOoSPXtARzBV,0QSPohbvsYOEcvkHnok13V,5YnNWHKCJaJKwbagXDb5Uf,7n3WO6ESKS1uCI9fgkGs66,6kAOsnRUgp21bPiUoVZeuJ,1uOe9m9bAiA

HTTP Error for GET to https://api.spotify.com/v1/tracks/?ids=6JJhNMv6JooXnN9rq9TBmZ,3U9Kkv90QElks0pzIgfAxw,4iUTK39pDuDaSBLAnkkkfQ,2tofXDrli8EbbsM7bZflXq,47y8hRlWLEJ4VafE6LMjEZ,7FMXSSzIRW8aJwUPfzXsa2,7w9s71v9Q5ifH46apOqA07,6xNwKNYZcvgV3XTIwsgNio,1ZzRuFUIYVIMQmPhZbnpdf,5gAuZSGGbtk1SNoJGFhRLK,11Ojp7JniVvwd0gmgvyKkd,4eAwB5pnKFTmsgc3zWoYO0,28ewfZ6bTCVCNOFEZ1Q7q8,5FZxsHWIvUsmSK1IAvm2pp,0qgrrDnUUhyxpxbBznUnzg,6rsoBvxrlxdmqJyGPPciyq,3QZ7uX97s82HFYSmQUAN1D,2gscB6kDOmrv1P6qs2KXE3,1KTJmfwrk5pYqsi9mkY3nT,4eMjf9FgbmFQ6yHZD5EsgX,1j0MYPwvprO2AVD8O2aGyg,5tCEbEcupMpNzgWcZyLnl0,4HCuWaxAx0YdFGTMin1OYa,1KuNk1ZZPTZKwjjo1Vdw9G,6Fg5ZX12c2clfZ4X47g0pj,3C9mS6yoyMzLbUV67X2zgj,7HBkWVzDkwFHEG2foE3AML,6IfGWBcMcb4MYMw15MZefz,24BWNB3oP5jDaogm2OgGJR,2OnST9dGaO88MNQvZ4VjaO,2RvluPimiBzerjtXE87AdI,4bCd2UsMl1l6g7HpuA53NG,6prvMx5SVgsFf8Iocaxvqz,4wwUsbKQ4zosUj6DC4vkjW,6tGyLydK5lNXz8Aw44wRj8,651aMl2jYthoJyI9WvNKo4,3PRUVpbUrGHh3Wl3lIYEyF,0V7cyyQLdJg0ythOljTCPn,0DpNNwOcvYQcGdGNezAPN8,5awljpWNO5TpXCyjpvCBbs,2gpFhHwBF5IZwP61MaM

[ERROR] Batch 100 failed: http status: 403, code: -1 - https://api.spotify.com/v1/tracks/?ids=6JJhNMv6JooXnN9rq9TBmZ,3U9Kkv90QElks0pzIgfAxw,4iUTK39pDuDaSBLAnkkkfQ,2tofXDrli8EbbsM7bZflXq,47y8hRlWLEJ4VafE6LMjEZ,7FMXSSzIRW8aJwUPfzXsa2,7w9s71v9Q5ifH46apOqA07,6xNwKNYZcvgV3XTIwsgNio,1ZzRuFUIYVIMQmPhZbnpdf,5gAuZSGGbtk1SNoJGFhRLK,11Ojp7JniVvwd0gmgvyKkd,4eAwB5pnKFTmsgc3zWoYO0,28ewfZ6bTCVCNOFEZ1Q7q8,5FZxsHWIvUsmSK1IAvm2pp,0qgrrDnUUhyxpxbBznUnzg,6rsoBvxrlxdmqJyGPPciyq,3QZ7uX97s82HFYSmQUAN1D,2gscB6kDOmrv1P6qs2KXE3,1KTJmfwrk5pYqsi9mkY3nT,4eMjf9FgbmFQ6yHZD5EsgX,1j0MYPwvprO2AVD8O2aGyg,5tCEbEcupMpNzgWcZyLnl0,4HCuWaxAx0YdFGTMin1OYa,1KuNk1ZZPTZKwjjo1Vdw9G,6Fg5ZX12c2clfZ4X47g0pj,3C9mS6yoyMzLbUV67X2zgj,7HBkWVzDkwFHEG2foE3AML,6IfGWBcMcb4MYMw15MZefz,24BWNB3oP5jDaogm2OgGJR,2OnST9dGaO88MNQvZ4VjaO,2RvluPimiBzerjtXE87AdI,4bCd2UsMl1l6g7HpuA53NG,6prvMx5SVgsFf8Iocaxvqz,4wwUsbKQ4zosUj6DC4vkjW,6tGyLydK5lNXz8Aw44wRj8,651aMl2jYthoJyI9WvNKo4,3PRUVpbUrGHh3Wl3lIYEyF,0V7cyyQLdJg0ythOljTCPn,0DpNNwOcvYQcGdGNezAPN8,5awljpWNO

...

[ERROR] Batch 900 failed: http status: 403, code: -1 - https://api.spotify.com/v1/tracks/?ids=49tWEHMTXFDMT9tpLlS008,74BkNu4vhqOLMi8hcJbtnm,1Zq8qQGFIKcZLWhnpchJZn,5E0HsaeTON2ucXXArmSHNp,2MIZMR2WHqtdHpeuKUcs1l,2hGj5iz7i5Vsle1O9QYGol,3phaRmdQXoC0UqSnM4EBrI,0ql727m0Dm5hszWIsI1BwX,1mgfms0E0uJhULWwqMcqtI,1e6sJSsSVQDq4ucpJ6l7cG,15OCqNPYoLziEAsbVnqRj5,4OIRWMwEHR6oCU7g1QgK0n,7fZiHuZ3OapgnshQ7QtxWV,2QFyOnJfJ49OnNgMvKXvTM,6nmukSDAmw3XtbGKGD8056,1I2zZtHCrBkoTgfwxnNKZ9,0HLWvLKQWpFdPhgk6ym58n,1fDsrQ23eTAVFElUMaf38X,2AEjVbLUDn3knYqIyxbsjI,6je1dnckDvoE04VJtC5zww,08VZ3ycsCq1LfabZjdRLg4,1x0JkipO1X78OC1OOxpOoM,0l8p1TkaVgcwIUS0xwm6Y5,0KQRhak4Irr2CKxgQ9U6ay,1mEDRi6QFAASefQpHUxajr,13H3jef8bfXNHUab8NOem3,2jHEySxCzzODFWmPzt084L,0R3LjmIZg8MiBlf3LdYjcD,1T7Tqsfkz0Ntbwta2hHebY,7MlPySldFcJ9R3mSFvGV4y,1wflx5YklLYToaIbXkrZSq,68HbD7rEFTlnJy4UWSpySY,1CtfA9JkJ5cJQgpO7wEj4V,27hFQQS3cVUmIK3ser5bpu,5hVKGw1OXBCEZgRWFUilk7,2PIlBukQ6limukVR8Ubb5o,1wja0kEF3DvlDDHUi060Bk,0kpCfLJbmYFybCO6xy0hBn,0qeKzbUsW0V4ZWRJrHNiD3,5fwSHlTEW

HTTP Error for GET to https://api.spotify.com/v1/tracks/?ids=2eYJSfG4XE5M1PbcecUiMd,6hZQGrJlAQwrNnUyFkdO8r,4PSITiLRcdZI5WyGlXAssT,5GDtkgG9T1BDknHHyDtghv,1QmgrcZquzAfW1OMALh1yb,188TDPhgzmi4Aul27nWlEf,3ZucFCxCGG9Jv1xuPJQF6t,3NyYDbTIBwCTiW0svrcVjy,1TuWr81XEnKF2oyKxeYHWf,0CbNE0yI2EZv9uIwXnP4p5,2iwuZ15yjlWFvd4pEyiWAr,3NGkNPtXtTXV5VqiLpGgLd,5gXeGIJCGNycRlkoBIpBeJ,6w1T511GDmnHWF1EqqZm2y,119ZRPeJ540NUGoUD8LkMe,5aJfbWiKcwC4Fh7ymbKHHy,0Pa0xSCRVO8ZIxs29XgBvv,7I1nwZimETiRW83xZv023W,2nNaw1QUcqiEX6pBFxcpp3,5LhYDk8wnGGeaQufZAe7ce,3lsaCXY238dhR9GW9DEV5e,4mYrbFGkk5u3bnDhTj0jAO,61tTaZhoX9wHV7ofCbsTDv,5L70RGPJEdooqtrgR7vd6U,7aokiMHeHTBynFXaiBlXbm,5hTeAOSQgpSjLeRD6qVAdQ,3RbAhVb297X0g0dQVN03L1,1rZwIk3LxrMmVyXghIYubr,3ibVGBrJ8owRcmBynolCum,5vsqY1n8gOnkKvaGQjShpm,54bICsSSkjGWpUTLmtfAYK,4RSxWEwSQKsO9C90LO0wsN,5gYNIwz1eygjeDnVCV8sCP,2WBqBvlsAmSGbp9AkRas3T,76zoOB2W5KAh77cFcvKZyJ,2xVoBE8IwIF3d7lpFE75iC,2R0W3yzaYLWOz1pDzUpgkh,5708ZWpCPG8bTtbrcIPvGx,01mu4LFiBrFkJyELag2eZe,4wD1Vpnw2jUHs07HeAEeFt,1gxXYbu3reRQZMCuh5j

[ERROR] Batch 1000 failed: http status: 403, code: -1 - https://api.spotify.com/v1/tracks/?ids=2eYJSfG4XE5M1PbcecUiMd,6hZQGrJlAQwrNnUyFkdO8r,4PSITiLRcdZI5WyGlXAssT,5GDtkgG9T1BDknHHyDtghv,1QmgrcZquzAfW1OMALh1yb,188TDPhgzmi4Aul27nWlEf,3ZucFCxCGG9Jv1xuPJQF6t,3NyYDbTIBwCTiW0svrcVjy,1TuWr81XEnKF2oyKxeYHWf,0CbNE0yI2EZv9uIwXnP4p5,2iwuZ15yjlWFvd4pEyiWAr,3NGkNPtXtTXV5VqiLpGgLd,5gXeGIJCGNycRlkoBIpBeJ,6w1T511GDmnHWF1EqqZm2y,119ZRPeJ540NUGoUD8LkMe,5aJfbWiKcwC4Fh7ymbKHHy,0Pa0xSCRVO8ZIxs29XgBvv,7I1nwZimETiRW83xZv023W,2nNaw1QUcqiEX6pBFxcpp3,5LhYDk8wnGGeaQufZAe7ce,3lsaCXY238dhR9GW9DEV5e,4mYrbFGkk5u3bnDhTj0jAO,61tTaZhoX9wHV7ofCbsTDv,5L70RGPJEdooqtrgR7vd6U,7aokiMHeHTBynFXaiBlXbm,5hTeAOSQgpSjLeRD6qVAdQ,3RbAhVb297X0g0dQVN03L1,1rZwIk3LxrMmVyXghIYubr,3ibVGBrJ8owRcmBynolCum,5vsqY1n8gOnkKvaGQjShpm,54bICsSSkjGWpUTLmtfAYK,4RSxWEwSQKsO9C90LO0wsN,5gYNIwz1eygjeDnVCV8sCP,2WBqBvlsAmSGbp9AkRas3T,76zoOB2W5KAh77cFcvKZyJ,2xVoBE8IwIF3d7lpFE75iC,2R0W3yzaYLWOz1pDzUpgkh,5708ZWpCPG8bTtbrcIPvGx,01mu4LFiBrFkJyELag2eZe,4wD1Vpnw

HTTP Error for GET to https://api.spotify.com/v1/tracks/?ids=1OAp6qN5KmoGUQ2edICKsC,5L7VBYoosmkmiiDlzumdCe,71EGDqRmdmC1Y4w5ZNM14B,0iujTWTgmqgrc6bXxVu9aC,1sIIlVrnPhrvmTrHtzM7tV,7AlSUZpdxkzvHJBut5rzjn,683p6Ov3orc4ICLOKi1Yn6,0FuFovPx3x77CRFtt22Q3H,4LUJzSLpBtgH3dpOH7J7Nf,72AlDuv7nkBb9DvWPNgIw1,0YFvHkxJVGYLioe26UH7xv,1ug41NniaCmW7Y2Lm8k1SY,3sFdaHo9D3hfiFt2wVi6a5,708feUe50Ed8N6zilSiSQM,5yrU9jSt6hGxodbRgXaaqa,1CxOnGWoUmxqlyebvPo3pZ,20DigsfRILHYKeMqbuDqtx,46o19o8yQHxnnjROdgoLxW,0kkqItjhDR2KLe3p9AUnRk,40Ffnc1HjnsZbWWGl8lTDq,0fRUwcyUiNm6KTITio97gJ,5C3X6QpPGGuFIlvwM8Eh0B,1OO6V10Lyxw0Mz5HJd4BWO,1VNB8G2Rq3RcQgo74OCHzp,5sXJukxMUBd2CX8nmwYHbD,3y3thO9GCZxsauTac1Hv1s,2zMFWinkVpu2dlm4QMR4Oq,6In2IjmE06RxlCSPpVizTQ,0Ex1dDvV8PW1B4p04NkM7C,4oaLPF7Y6udoeuxmqJzfiD,1g54Kfpkw2ZetZdRXdUmaD,1t5X4GCncICh7BmMbGDYLf,4VX39ulMXYlgBivW2UHdze,2XQzNolwlP2eH2ejLlUcem,6gDhsPo1qR8ZoZiOon2Isc,07XXH6BByArHCUym1bvALj,0oiWvwSs3jY00AIwK4vas6,1s1DUBMX8MzwPeIKcP4LUU,6gvzZ0E4uTFXRqCQDc8Z0R,3F9QPgLzUXDIgbLdRu1rSq,5eq83Qv4nFwht3tL3UC

[ERROR] Batch 1100 failed: http status: 403, code: -1 - https://api.spotify.com/v1/tracks/?ids=1OAp6qN5KmoGUQ2edICKsC,5L7VBYoosmkmiiDlzumdCe,71EGDqRmdmC1Y4w5ZNM14B,0iujTWTgmqgrc6bXxVu9aC,1sIIlVrnPhrvmTrHtzM7tV,7AlSUZpdxkzvHJBut5rzjn,683p6Ov3orc4ICLOKi1Yn6,0FuFovPx3x77CRFtt22Q3H,4LUJzSLpBtgH3dpOH7J7Nf,72AlDuv7nkBb9DvWPNgIw1,0YFvHkxJVGYLioe26UH7xv,1ug41NniaCmW7Y2Lm8k1SY,3sFdaHo9D3hfiFt2wVi6a5,708feUe50Ed8N6zilSiSQM,5yrU9jSt6hGxodbRgXaaqa,1CxOnGWoUmxqlyebvPo3pZ,20DigsfRILHYKeMqbuDqtx,46o19o8yQHxnnjROdgoLxW,0kkqItjhDR2KLe3p9AUnRk,40Ffnc1HjnsZbWWGl8lTDq,0fRUwcyUiNm6KTITio97gJ,5C3X6QpPGGuFIlvwM8Eh0B,1OO6V10Lyxw0Mz5HJd4BWO,1VNB8G2Rq3RcQgo74OCHzp,5sXJukxMUBd2CX8nmwYHbD,3y3thO9GCZxsauTac1Hv1s,2zMFWinkVpu2dlm4QMR4Oq,6In2IjmE06RxlCSPpVizTQ,0Ex1dDvV8PW1B4p04NkM7C,4oaLPF7Y6udoeuxmqJzfiD,1g54Kfpkw2ZetZdRXdUmaD,1t5X4GCncICh7BmMbGDYLf,4VX39ulMXYlgBivW2UHdze,2XQzNolwlP2eH2ejLlUcem,6gDhsPo1qR8ZoZiOon2Isc,07XXH6BByArHCUym1bvALj,0oiWvwSs3jY00AIwK4vas6,1s1DUBMX8MzwPeIKcP4LUU,6gvzZ0E4uTFXRqCQDc8Z0R,3F9QPgLz

In [15]:
      #spotify verions[IGNORE]

import requests
import io
import soundfile as sf

def download_audio(url: str) -> tuple[np.ndarray, int] | None:
    """Download preview MP3 and return (audio array, sample_rate)."""
    try:
        response = requests.get(url, timeout=10)
        audio, sr = sf.read(io.BytesIO(response.content))
        # Convert stereo to mono if needed
        if audio.ndim == 2:
            audio = audio.mean(axis=1)
        return audio.astype(np.float32), sr
    except Exception as e:
        print(f"[ERROR] Download failed: {e}")
        return None


def extract_mfcc(audio: np.ndarray, sr: int, n_mfcc: int = 13) -> np.ndarray:
    """
    Extract MFCCs and return a (n_mfcc * 3,) feature vector:
    mean, std, and delta-mean for each coefficient.
    """
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)        # (13, T)
    delta = librosa.feature.delta(mfcc)                                  # (13, T)

    features = np.concatenate([
        mfcc.mean(axis=1),    # mean of each coefficient over time
        mfcc.std(axis=1),     # std  of each coefficient over time
        delta.mean(axis=1),   # mean of delta (captures temporal change)
    ])
    return features  # shape: (39,)


def build_mfcc_dataset(df: pd.DataFrame, n_mfcc: int = 13) -> pd.DataFrame:
    records = []
    no_preview = 0

    for _, row in df.iterrows():
        if pd.isna(row['preview_url']):
            no_preview += 1
            continue

        result = download_audio(row['preview_url'])
        if result is None:
            continue

        audio, sr = result
        features = extract_mfcc(audio, sr, n_mfcc=n_mfcc)

        # Column names: mfcc_mean_1..13, mfcc_std_1..13, mfcc_delta_1..13
        feature_cols = (
            [f'mfcc_mean_{i+1}' for i in range(n_mfcc)] +
            [f'mfcc_std_{i+1}'  for i in range(n_mfcc)] +
            [f'mfcc_delta_{i+1}'for i in range(n_mfcc)]
        )

        record = {
            'track_id':    row['track_id'],
            'track_genre': row['track_genre'],
            'genre':       row['genre'],
            **dict(zip(feature_cols, features))
        }
        records.append(record)

    print(f"Extracted MFCCs for {len(records)} tracks. Skipped {no_preview} (no preview).")
    return pd.DataFrame(records)


df_previews = pd.read_csv('sampled_tracks_with_previews.csv')
mfcc_df = build_mfcc_dataset(df_previews)
mfcc_df.to_csv('mfcc_features.csv', index=False)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 10)

In [16]:
# Using YT to extract sngs

from typing import Optional

def get_audio_clip(track_name: str, artist: str, duration_ms: int = None, out_dir: str = 'audio_clips') -> Optional[str]:
    os.makedirs(out_dir, exist_ok=True)
    query = f"{track_name} {artist} official audio"
    safe_name = f"{track_name[:40]}_{artist[:20]}".replace('/', '_').replace(' ', '_')
    out_path = os.path.join(out_dir, safe_name)

    if duration_ms:
        mid = (duration_ms / 1000) // 2
        start = int(mid - 15)
        end   = int(mid + 15)
    else:
        start, end = 60, 90

    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': out_path + '.%(ext)s',
        'quiet': True,
        'no_warnings': True,
        'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'wav'}],
        'download_sections': {f'*{start}-{end}': True},
        'default_search': 'ytsearch1',
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([query])
        return out_path + '.wav'
    except Exception as e:
        print(f"[ERROR] {track_name} - {artist}: {e}")
        return None


def extract_mfcc(filepath: str, n_mfcc: int = 13) -> Optional[np.ndarray]:
    try:
        audio, sr = librosa.load(filepath, sr=22050, duration=30)
        mfcc      = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        delta     = librosa.feature.delta(mfcc)
        return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1), delta.mean(axis=1)])
    except Exception as e:
        print(f"[ERROR] MFCC extraction failed for {filepath}: {e}")
        return None



In [17]:
#test
test_row = sampled_df.iloc[0]
print(f"Testing: {test_row['track_name']} — {test_row['artists']}")

filepath = get_audio_clip(test_row['track_name'], test_row['artists'], duration_ms=test_row['duration_ms'])
print(f"Audio saved to: {filepath}")

features = extract_mfcc(filepath)
print(f"MFCC shape: {features.shape}")
print(f"First few values: {features[:5]}")

Testing: parents — YUNGBLUD
Audio saved to: audio_clips/parents_YUNGBLUD.wav           
MFCC shape: (39,)
First few values: [-45.296093   66.90737     7.7488155  14.39976     5.574636 ]


In [19]:
def build_mfcc_dataset(df: pd.DataFrame, n_mfcc: int = 13) -> pd.DataFrame:
    records = []
    feature_cols = (
        [f'mfcc_mean_{i+1}'  for i in range(n_mfcc)] +
        [f'mfcc_std_{i+1}'   for i in range(n_mfcc)] +
        [f'mfcc_delta_{i+1}' for i in range(n_mfcc)]
    )

    total = len(df)
    for i, (_, row) in enumerate(df.iterrows()):
        print(f"[{i+1}/{total}] {row['track_name']} — {row['artists']}")

        filepath = get_audio_clip(row['track_name'], row['artists'], duration_ms=row['duration_ms'])
        if not filepath or not os.path.exists(filepath):
            print("  ↳ skipped (no audio)")
            continue

        features = extract_mfcc(filepath)
        os.remove(filepath)

        if features is None:
            print("  ↳ skipped (MFCC failed)")
            continue

        records.append({
            'track_id':    row['track_id'],
            'track_genre': row['track_genre'],
            'genre':       row['genre'],
            **dict(zip(feature_cols, features))
        })

        # Checkpoint every 50 tracks
        if len(records) % 50 == 0:
            pd.DataFrame(records).to_csv('mfcc_features_checkpoint.csv', index=False)
            print(f"  ✓ checkpoint saved ({len(records)} tracks so far)")

    mfcc_df = pd.DataFrame(records)
    mfcc_df.to_csv('mfcc_features.csv', index=False)
    print(f"\nDone! Extracted MFCCs for {len(records)}/{total} tracks.")
    return mfcc_df

In [21]:
# running for full mfcc
mfcc_df = build_mfcc_dataset(sampled_df)


[1/1164] parents — YUNGBLUD
[2/1164] Merry Christmas, Kiss My Ass — All Time Low     
[3/1164] Jingle Bell Rock — George Strait                
[4/1164] Secrets — OneRepublic                           
[5/1164] Es Por Ti — Juanes                              
[6/1164] Winter Wonderland — Rod Stewart;Michael Bublé   
[7/1164] If I Had You — Rod Stewart                      
[8/1164] Thunderstruck — AC/DC                             
[9/1164] I Belong To You — Lenny Kravitz                   
[10/1164] Black Summer — Red Hot Chili Peppers           
[11/1164] Finish Line — Skillet                          
[12/1164] Won't Back Down — Fuel                         
[13/1164] Ciencia De La Lluvia — Enjambre                
[14/1164] Living Hope — Phil Wickham                     
[15/1164] Middle Finger — Bohnes                           


ERROR: [youtube] ohocp8JWbGQ: This video is not available


[ERROR] Middle Finger - Bohnes: ERROR: [youtube] ohocp8JWbGQ: This video is not available
  ↳ skipped (no audio)
[16/1164] El Recuento de los Daños — Los Estramboticos
[17/1164] In The Name Of Love - Recorded At Spotify Studios NYC — Kari Jobe
[18/1164] Awake and Alive — Skillet                      
[19/1164] Playground (from the series Arcane League of Legends) — Bea Miller
[20/1164] Symphony Of Destruction — Megadeth             
[21/1164] Punk Superstars — GUFI                           
[22/1164] Heute hier, morgen dort - Live — Die Toten Hosen
[23/1164] Pretty Fly (For A White Guy) — The Offspring   
[24/1164] Beat the Devil's Tattoo — Black Rebel Motorcycle Club
[25/1164] Adela en el Carrousell — Charly García         
[26/1164] Amnesia — Say Ocean                            
[27/1164] Você Sabe — Autoramas                           
[28/1164] 80's — Allison                                 
[29/1164] Always — blink-182                             
[30/1164] Spellbound — Siouxsi

ERROR: [youtube] iex6eui5o0s: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


[ERROR] Fat Fuck - CKY: ERROR: [youtube] iex6eui5o0s: Sign in to confirm your age. This video may be inappropriate for some users. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies
  ↳ skipped (no audio)
[197/1164] Fatal Fall — Flaw
[198/1164] Snap Your Fingers, Snap Your Neck — Prong       
[199/1164] Bender — Sevendust                            
[200/1164] Electric Avenue — Powerman 5000               
[201/1164] Tie Me Down (with Elley Duhé) — Gryffin;Elley Duhé
[202/1164] Hollywood Perfect — Unknown Brain;NotEvenTanner 
[203/1164] The Bender — Matoma;Brando                      
[204/1164] Poison — Rita Ora                               
  ✓ checkpoint saved (200 tracks so far)                   
[205/1164] Takeaway (feat

...

[ERROR] Wake Up - Christopher Zondaflex Tyler: ERROR: [youtube] Yo1P0u_1xes: This video is not available
  ↳ skipped (no audio)
[1132/1164] One Two Buckle My Shoe — Kidsounds


ERROR: [youtube] eT4BwDl7yHI: This video is not available


[ERROR] One Two Buckle My Shoe - Kidsounds: ERROR: [youtube] eT4BwDl7yHI: This video is not available
  ↳ skipped (no audio)
[1133/1164] Brush Your Teeth — Desmond Dennis
[1134/1164] Daddy Minion — Desmond Dennis                


ERROR: [youtube] OLV02G_83YI: This video is not available


[ERROR] Daddy Minion - Desmond Dennis: ERROR: [youtube] OLV02G_83YI: This video is not available
  ↳ skipped (no audio)
[1135/1164] I Killed a Baby — Christopher Titus
[1136/1164] Country Music / Mexican Cowboy / Tough Cowboy — Pablo Francisco
[1137/1164] The Spirit Cave — Patton Oswalt              
[1138/1164] Snootch — Iliza Shlesinger                   
[1139/1164] A Basket of Kisses — Greg Proops               
[1140/1164] Oscar Movies and Paula Deen — D.L. Hughley     
[1141/1164] New York to LA — Andy Hendrickson              
[1142/1164] My Rehab Counselor — Artie Lange             
[1143/1164] Brittni and Judah — Lil Rel Howery           
  ↳ skipped (no audio)
[1144/1164] Wedding Story — Mike Birbiglia
[1145/1164] I fucked up — convolk                          
[1146/1164] the overthinker — One Hope                   
[1147/1164] circus — PmBata                              
[1148/1164] used to — Josh Golden                        
[1149/1164] Ease Off — YNG Martyr           